# FFT Analysis (Analysis 1 & 2)
TN/FP/TP/FN 그룹별 2D FFT 히트맵 및 Radial Profile 분석

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
from scipy.stats import linregress
from tqdm import tqdm

## 설정

In [ ]:
# [설정] TN/FP/TP/FN 폴더가 들어있는 루트
CASE_DIR = "/home/ice06/project/HS/Dataset_0223/Case2"
# [설정] 결과 저장 폴더
OUTPUT_DIR = "./analysis_FFT_results_TNFP_TPFN"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"결과 저장 폴더: {OUTPUT_DIR}")

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff"}
GROUPS = ["TN", "FP", "TP", "FN"]

## 이미지 파일 로드

In [ ]:
def get_image_files_TNFP_TPFN(case_dir: str):
    """
    case_dir/TN, case_dir/FP, case_dir/TP, case_dir/FN 하위 이미지를 읽어옴
    """
    data_list = []
    for g in GROUPS:
        gdir = os.path.join(case_dir, g)
        if not os.path.isdir(gdir):
            print(f"⚠️ 폴더 없음: {gdir}")
            continue

        for fname in os.listdir(gdir):
            ext = os.path.splitext(fname)[1].lower()
            if ext not in IMG_EXT:
                continue
            data_list.append({"path": os.path.join(gdir, fname), "label": g})

    data_list.sort(key=lambda x: (x["label"], x["path"]))
    return data_list

In [ ]:
if not os.path.isdir(CASE_DIR):
    raise FileNotFoundError(f"CASE_DIR not found: {CASE_DIR}")

data_info = get_image_files_TNFP_TPFN(CASE_DIR)
print(f"총 {len(data_info)}장의 이미지를 찾았습니다.")

found = sorted(list(set([d["label"] for d in data_info])))
print(f"발견된 그룹: {found}")

if len(data_info) == 0:
    print("❌ 이미지가 없습니다.")

## Analysis 1 : 2D FFT 평균 히트맵 (TN/FP/TP/FN)

In [ ]:
def analyze_visual_fft_grouped(data_list, img_size=512):
    grouped = {g: [] for g in GROUPS}
    for item in data_list:
        if item["label"] in grouped:
            grouped[item["label"]].append(item["path"])

    print(f"\n[Analysis 1] 2D FFT Heatmap (TN/FP/TP/FN)...")

    n = len(GROUPS)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))

    for idx, g in enumerate(GROUPS):
        paths = grouped[g]
        avg_mag = np.zeros((img_size, img_size), dtype=np.float64)
        count = 0

        for path in tqdm(paths, desc=f"FFT processing ({g})"):
            try:
                img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    continue
                img = cv2.resize(img, (img_size, img_size))

                f = np.fft.fft2(img)
                fshift = np.fft.fftshift(f)
                mag = 20 * np.log(np.abs(fshift) + 1e-9)

                avg_mag += mag
                count += 1
            except Exception:
                continue

        ax = axes[idx]
        if count > 0:
            avg_mag /= count
            im = ax.imshow(avg_mag, cmap="inferno")
            ax.set_title(f"{g}\n(N={count})")
            ax.axis("off")
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        else:
            ax.set_title(f"{g}\n(N=0)")
            ax.axis("off")

    save_path = os.path.join(OUTPUT_DIR, "1_FFT_Heatmap_TNFP_TPFN.png")
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f">> 저장 완료: {save_path}")

In [ ]:
analyze_visual_fft_grouped(data_info, img_size=512)

## Analysis 2 : Radial Profile 기반 Log-Log Slope (TN/FP/TP/FN)

In [ ]:
def get_radial_profile(image):
    h, w = image.shape
    center = (w // 2, h // 2)
    y, x = np.ogrid[:h, :w]
    r = np.sqrt((x - center[0]) ** 2 + (y - center[1]) ** 2).astype(int)

    f = np.fft.fft2(image)
    fshift = np.fft.fftshift(f)
    mag = np.abs(fshift)

    tbin = np.bincount(r.ravel(), weights=mag.ravel())
    nr = np.bincount(r.ravel())
    return tbin / (nr + 1e-9)

In [ ]:
def analyze_fft_numeric_grouped(data_list, img_size=512, start_freq=10, end_frac=0.8):
    """
    TN/FP/TP/FN 각 그룹 평균 radial profile -> log-log plot + slope 계산
    """
    grouped = {g: [] for g in GROUPS}
    for item in data_list:
        if item["label"] in grouped:
            grouped[item["label"]].append(item["path"])

    print(f"\n[Analysis 2] Frequency Energy Distribution (TN/FP/TP/FN)...")

    color_map = {
        "TN": "#2ecc71",  # green
        "FP": "#f39c12",  # orange
        "TP": "#3498db",  # blue
        "FN": "#e74c3c",  # red
    }

    plt.figure(figsize=(10, 8))

    slopes = {}

    for g in GROUPS:
        paths = grouped[g]
        avg_profile = None
        count = 0

        for path in tqdm(paths, desc=f"Profile calc ({g})"):
            try:
                img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    continue
                img = cv2.resize(img, (img_size, img_size))
                profile = get_radial_profile(img)

                if avg_profile is None:
                    avg_profile = profile
                else:
                    length = min(len(avg_profile), len(profile))
                    avg_profile = avg_profile[:length] + profile[:length]

                count += 1
            except Exception:
                continue

        if count <= 0 or avg_profile is None:
            print(f"⚠️ {g}: 유효 이미지 없음")
            continue

        avg_profile /= count

        end_freq = int(len(avg_profile) * end_frac)
        if end_freq <= start_freq + 5:
            print(f"⚠️ {g}: 주파수 구간이 너무 짧음")
            continue

        freqs = np.arange(start_freq, end_freq)
        power = avg_profile[start_freq:end_freq]

        log_f = np.log(freqs)
        log_p = np.log(power + 1e-9)

        slope, intercept, _, _, _ = linregress(log_f, log_p)
        slopes[g] = slope

        plt.plot(
            log_f,
            log_p,
            color=color_map.get(g, "gray"),
            alpha=0.85,
            linewidth=2,
            label=f"{g} (N={count}, Slope={slope:.2f})"
        )

    plt.title("Frequency Energy Distribution by Group (TN/FP/TP/FN)")
    plt.xlabel("Log Frequency")
    plt.ylabel("Log Power")
    plt.legend()
    plt.grid(True, alpha=0.3)

    save_path = os.path.join(OUTPUT_DIR, "2_FFT_Profile_Comparison_TNFP_TPFN.png")
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f">> 저장 완료: {save_path}")

    slope_txt = os.path.join(OUTPUT_DIR, "2_FFT_Slopes_TNFP_TPFN.txt")
    with open(slope_txt, "w", encoding="utf-8") as f:
        for g in GROUPS:
            if g in slopes:
                f.write(f"{g}\t{slopes[g]:.6f}\n")
    print(f">> slope 저장 완료: {slope_txt}")

In [ ]:
analyze_fft_numeric_grouped(data_info, img_size=512, start_freq=10, end_frac=0.8)
print(f"\n[완료] 모든 분석 결과가 '{OUTPUT_DIR}'에 저장되었습니다.")